In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

laps_table = f"{catalog}.f1_silver.lap_times"
pits_table = f"{catalog}.f1_silver.pit_stops"

laps_df = spark.table(laps_table)
pits_df = spark.table(pits_table)

In [0]:
pit_boundaries_df = pits_df.select(
    "race_id",
    "driver_id",
    "pit_lap"
)

laps_marked_df = (
    laps_df.alias("laps")
    .join(
        pit_boundaries_df.alias("pits"),
        (
            (F.col("laps.race_id") == F.col("pits.race_id"))
            & (F.col("laps.driver_id") == F.col("pits.driver_id"))
            & (F.col("laps.lap_number") == F.col("pits.pit_lap") + 1)
        ),
        "left"
    )
    .select(
        F.col("laps.*"),
        F.when(
            F.col("pits.pit_lap").isNotNull(),
            1
        ).otherwise(0).alias("new_stint")
    )
)

In [0]:
stint_window = (
    Window
    .partitionBy("race_id", "driver_id")
    .orderBy("lap_number")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

laps_with_stints_df = (
    laps_marked_df
    .withColumn(
        "stint_number",
        F.sum("new_stint").over(stint_window) + 1
    )
)

display(
    laps_with_stints_df
    .select(
        "race_id",
        "driver_id",
        "lap_number",
        "lap_time_ms",
        "stint_number"
    )
    .orderBy("driver_id", "lap_number")
)

In [0]:
lap_order_window = (
    Window
    .partitionBy("race_id", "driver_id", "stint_number")
    .orderBy("lap_number")
)

laps_enriched_df = (
    laps_with_stints_df
    .withColumn(
        "lap_in_stint",
        F.row_number().over(lap_order_window)
    )
)

In [0]:
stint_metrics_df = (
    laps_enriched_df
    .groupBy(
        "race_id",
        "driver_id",
        "stint_number"
    )
    .agg(
        F.min("lap_number").alias("stint_start_lap"),
        F.max("lap_number").alias("stint_end_lap"),
        F.count("*").alias("stint_laps"),
        F.avg("lap_time_ms").alias("average_lap_time_ms"),
        F.min("lap_time_ms").alias("best_lap_time_ms"),
        F.max("lap_time_ms").alias("slowest_lap_time_ms")
    )
    .withColumn(
        "tyre_compound",
        F.lit("UNKNOWN")
    )
)

display(
    stint_metrics_df
    .orderBy("driver_id", "stint_number")
)

In [0]:
gold_table = f"{catalog}.f1_gold.driver_stint_metrics"

(
    stint_metrics_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Saved Gold table: {gold_table}")

In [0]:
display(
    spark.table(gold_table)
    .orderBy("race_id", "driver_id", "stint_number")
)

In [0]:
catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

gold_metrics_table = f"{catalog}.f1_gold.driver_stint_metrics"
openf1_stints_table = f"{catalog}.f1_silver.openf1_tyre_stints"

stint_metrics_df = spark.table(gold_metrics_table)
compound_stints_df = spark.table(openf1_stints_table)

In [0]:
from pyspark.sql import functions as F

# Keep only the fields needed to enrich Gold.
compound_lookup_df = (
    compound_stints_df
    .select(
        "race_id",
        "driver_id",
        "stint_number",
        "tyre_compound",
        "stint_start_lap",
        "stint_end_lap"
    )
    .dropDuplicates(
        ["race_id", "driver_id", "stint_number"]
    )
)

In [0]:
gold_enriched_df = (
    stint_metrics_df
    .drop("tyre_compound")
    .join(
        compound_lookup_df,
        on=["race_id", "driver_id", "stint_number"],
        how="left"
    )
    .withColumn(
        "tyre_compound",
        F.coalesce(
            F.col("tyre_compound"),
            F.lit("UNKNOWN")
        )
    )
)

In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

gold_metrics_table = f"{catalog}.f1_gold.driver_stint_metrics"
openf1_stints_table = f"{catalog}.f1_silver.openf1_tyre_stints"

stint_metrics_df = spark.table(gold_metrics_table)
openf1_raw_df = spark.table(openf1_stints_table)

print("OpenF1 columns:", openf1_raw_df.columns)

In [0]:
current_race_id = (
    stint_metrics_df
    .select("race_id")
    .dropDuplicates()
    .first()["race_id"]
)

compound_lookup_df = (
    openf1_raw_df
    .withColumn("race_id", F.lit(current_race_id))
    .withColumnRenamed("Driver_id", "driver_id")
    .withColumnRenamed("Driver_code", "driver_code")
    .withColumnRenamed("compound", "tyre_compound")
    .select(
        "race_id",
        "driver_id",
        "driver_code",
        "stint_number",
        "tyre_compound"
    )
    .dropDuplicates(
        ["race_id", "driver_id", "stint_number"]
    )
)

display(compound_lookup_df)

In [0]:
gold_enriched_df = (
    stint_metrics_df
    .drop("tyre_compound")
    .join(
        compound_lookup_df,
        on=["race_id", "driver_id", "stint_number"],
        how="left"
    )
    .withColumn(
        "tyre_compound",
        F.coalesce(
            F.col("tyre_compound"),
            F.lit("UNKNOWN")
        )
    )
)

display(gold_enriched_df)

In [0]:
gold_table = f"{catalog}.f1_gold.driver_stint_metrics"

(
    gold_enriched_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_table)
)

print(f"Saved enriched Gold table: {gold_table}")

In [0]:
display(
    spark.table(gold_table)
    .groupBy("tyre_compound")
    .count()
    .orderBy(F.desc("count"))
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

laps_df = spark.table(
    f"{catalog}.f1_silver.lap_times"
)

pits_df = spark.table(
    f"{catalog}.f1_silver.pit_stops"
)

pit_boundaries_df = (
    pits_df
    .select("race_id", "driver_id", "pit_lap")
    .dropDuplicates()
)

laps_marked_df = (
    laps_df.alias("laps")
    .join(
        pit_boundaries_df.alias("pits"),
        (
            (F.col("laps.race_id") == F.col("pits.race_id"))
            & (F.col("laps.driver_id") == F.col("pits.driver_id"))
            & (F.col("laps.lap_number") == F.col("pits.pit_lap") + 1)
        ),
        "left"
    )
    .select(
        F.col("laps.*"),
        F.when(
            F.col("pits.pit_lap").isNotNull(),
            1
        ).otherwise(0).alias("new_stint")
    )
)

In [0]:
stint_window = (
    Window
    .partitionBy("race_id", "driver_id")
    .orderBy("lap_number")
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)

laps_with_stints_df = (
    laps_marked_df
    .withColumn(
        "stint_number",
        F.sum("new_stint").over(stint_window) + 1
    )
)

In [0]:
stint_order_window = (
    Window
    .partitionBy("race_id", "driver_id", "stint_number")
    .orderBy("lap_number")
)

stint_size_window = (
    Window
    .partitionBy("race_id", "driver_id", "stint_number")
)

laps_positioned_df = (
    laps_with_stints_df
    .withColumn(
        "lap_in_stint",
        F.row_number().over(stint_order_window)
    )
    .withColumn(
        "stint_laps",
        F.count("*").over(stint_size_window)
    )
)

In [0]:
degradation_df = (
    laps_positioned_df
    .groupBy(
        "race_id",
        "driver_id",
        "stint_number"
    )
    .agg(
        F.max("stint_laps").alias("stint_laps"),

        F.avg(
            F.when(
                F.col("lap_in_stint") <= 3,
                F.col("lap_time_ms")
            )
        ).alias("early_avg_lap_ms"),

        F.avg(
            F.when(
                F.col("lap_in_stint") > F.col("stint_laps") - 3,
                F.col("lap_time_ms")
            )
        ).alias("late_avg_lap_ms")
    )
    .filter(F.col("stint_laps") >= 6)
    .withColumn(
        "pace_delta_ms",
        F.col("late_avg_lap_ms") - F.col("early_avg_lap_ms")
    )
    .withColumn(
        "degradation_ms_per_lap",
        F.col("pace_delta_ms")
        / (F.col("stint_laps") - 1)
    )
)

display(degradation_df)

In [0]:
compound_reference_df = (
    spark.table(gold_table)
    .select(
        "race_id",
        "driver_id",
        "stint_number",
        "tyre_compound"
    )
    .dropDuplicates(
        ["race_id", "driver_id", "stint_number"]
    )
)

final_degradation_df = (
    degradation_df
    .join(
        compound_reference_df,
        on=["race_id", "driver_id", "stint_number"],
        how="left"
    )
    .withColumn(
        "tyre_compound",
        F.coalesce(
            F.col("tyre_compound"),
            F.lit("UNKNOWN")
        )
    )
)

display(final_degradation_df)

In [0]:
degradation_table = f"{catalog}.f1_gold.tyre_degradation"

(
    final_degradation_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(degradation_table)
)

print(f"Saved: {degradation_table}")

In [0]:
display(
    spark.table(degradation_table)
    .groupBy("tyre_compound")
    .agg(
        F.count("*").alias("stint_count"),
        F.avg("degradation_ms_per_lap").alias(
            "avg_degradation_ms_per_lap"
        ),
        F.avg("pace_delta_ms").alias(
            "avg_early_to_late_delta_ms"
        )
    )
    .orderBy("tyre_compound")
)